In [ ]:

!pip install transformers datasets scikit-learn evaluate -q
# !pip install --upgrade transformers

import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
import evaluate

#  Load already-saved train/val/test CSVs

train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/val.csv")
test_df = pd.read_csv("/content/test.csv")

# -------------------------------
# Convert labels to integers
# -------------------------------
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}

for df in [train_df, val_df, test_df]:
    # Clean strings (strip, lowercase)
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    # Convert to integers
    df["label"] = df["label"].map(label_mapping)

print("Unique labels in train:", train_df["label"].unique())

print("✅ Files loaded successfully")
print("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))

# Tokenizer and Dataset Conversion

model_name = "l3cube-pune/marathi-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["Text"], padding="max_length", truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize, batched=True)
test_dataset = Dataset.from_pandas(test_df).map(tokenize, batched=True)

for ds in [train_dataset, val_dataset, test_dataset]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
#
#  Load Model

num_labels = len(train_df["label"].unique())
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

#  Define Metrics

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    prec = precision_metric.compute(predictions=predictions, references=labels, average="macro")
    rec = recall_metric.compute(predictions=predictions, references=labels, average="macro")
    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"],
        "precision": prec["precision"],
        "recall": rec["recall"]
    }

#  Training Arguments


training_args = TrainingArguments(
    output_dir="./results_mahabert_v3",
    num_train_epochs=6,                # slightly more epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-5,                # a bit higher = faster learning
    warmup_steps=400,                  # slightly smoother start
    weight_decay=0.03,                 # slightly lower regularization
    max_grad_norm=1.0,                 # prevents exploding gradients
    logging_dir="./logs_mahabert_v3",
    logging_steps=1000,
    metric_for_best_model="f1",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train the Model

trainer.train()

#  Evaluate on Validation Set

val_results = trainer.evaluate(eval_dataset=val_dataset)
print("\n Validation Results:")
print(val_results)


#  Evaluate on Test Set

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("\n Test Results:")
print(test_results)


print("\n✅ Model training and evaluation complete.")

model.save_pretrained("./results_mahabert_v2/final_model")
tokenizer.save_pretrained("./results_mahabert_v2/final_model")


In [ ]:
# Install if needed
# !pip install gradio torch transformers pandas

import gradio as gr
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --------------------------------------------------------
# LOAD MODEL
# --------------------------------------------------------
model_path = "./results_mahabert_v2/final_model"
original_model_name = "l3cube-pune/marathi-bert"

tokenizer = AutoTokenizer.from_pretrained(original_model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

labels = ["Negative", "Neutral", "Positive"]
colors = {
    "Negative": "red",
    "Neutral": "orange",
    "Positive": "green",
    "Mixed": "purple"
}

# --------------------------------------------------------
# SAFE PREDICTION FUNCTION
# --------------------------------------------------------
def predict_sentiment(text):
    if not text or str(text).strip() == "":
        return "Neutral"  # fallback

    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]

    pred_idx = torch.argmax(probs).item()
    return labels[pred_idx]

# --------------------------------------------------------
# SINGLE TEXT ANALYSIS (NO CONFIDENCE)
# --------------------------------------------------------
def analyze_text(text):
    if not text or not text.strip():
        return "<span style='color:red;'>Please enter some text!</span>"

    sentiment = predict_sentiment(text)
    color = colors[sentiment]

    return f"""
    <div style='font-size:18px;'>
        <b>Sentiment:</b>
        <span style='color:{color}; font-weight:800;'>{sentiment}</span>
    </div>
    """

# --------------------------------------------------------
# CSV ANALYSIS (FIXED)
# --------------------------------------------------------
def analyze_csv(file, progress=gr.Progress()):
    if file is None:
        return None, "<span style='color:red;'>Upload a CSV first!</span>"

    # ✅ Robust CSV reading
    try:
        df = pd.read_csv(file.name, encoding="utf-8")
    except:
        try:
            df = pd.read_csv(file.name, encoding="latin-1")
        except Exception as e:
            return None, f"<span style='color:red;'>CSV Read Error: {str(e)}</span>"

    # ✅ Column check
    if "headline" not in df.columns:
        return None, "<span style='color:red;'>CSV must contain a 'headline' column!</span>"

    # ✅ Drop empty rows
    df = df.dropna(subset=["headline"])

    if len(df) == 0:
        return None, "<span style='color:red;'>CSV has no valid headlines!</span>"

    output = []
    sentiment_counter = {"Positive": 0, "Neutral": 0, "Negative": 0}

    total = len(df)

    # ✅ Process safely
    for i, row in df.iterrows():
        text = str(row["headline"])

        sentiment = predict_sentiment(text)

        sentiment_counter[sentiment] += 1
        output.append([text, sentiment])

        progress((i + 1) / total)

    result_df = pd.DataFrame(output, columns=["headline", "Sentiment"])

    # ------------------------------------------------
    # THRESHOLD LOGIC (CORRECT)
    # ------------------------------------------------
    total_count = sum(sentiment_counter.values())

    top_sentiment = max(sentiment_counter, key=sentiment_counter.get)
    top_percentage = (sentiment_counter[top_sentiment] / total_count) * 100

    threshold = 55

    if top_percentage >= threshold:
        overall_sentiment = top_sentiment
    else:
        overall_sentiment = "Mixed"

    color = colors[overall_sentiment]

    summary_html = f"""
    <div style='font-size:22px; font-weight:800; color:{color}; text-align:left; padding-left:10px;'>
        Overall Sentiment: {overall_sentiment}
    </div>
    """

    return result_df, summary_html

# --------------------------------------------------------
# UI DESIGN
# --------------------------------------------------------
custom_css = """
body { background: #f3f6fc !important; }
.gradio-container { font-family: 'Segoe UI'; }
#title { text-align:center; font-size: 34px; color:#007BFF; font-weight:900; }
.gr-button { background:#007BFF !important; color:white !important; border-radius:10px !important; }
.gr-box { border-radius:12px !important; }
"""

# --------------------------------------------------------
# UI
# --------------------------------------------------------
with gr.Blocks(css=custom_css) as app:

    gr.HTML("<div id='title'>Marathi News Sentiment Analyzer</div>")

    # SINGLE TEXT
    with gr.Tab("🔍 Analyze Single Text"):
        text_input = gr.Textbox(label="Enter Marathi Headline", lines=3)
        analyze_btn = gr.Button("Analyze")
        result_box = gr.HTML()

    # CSV
    with gr.Tab("📄 Analyze CSV File"):
        csv_input = gr.File(label="Upload CSV (must contain 'headline')")
        csv_output = gr.DataFrame()
        summary_output = gr.HTML()
        csv_analyze_btn = gr.Button("Process CSV")

    analyze_btn.click(analyze_text, inputs=text_input, outputs=result_box)

    csv_analyze_btn.click(
        analyze_csv,
        inputs=csv_input,
        outputs=[csv_output, summary_output]
    )

app.launch()